# 🤖 EchoNest Business Agent — EECE 503P/798S Assignment C3

**Milo**, EchoNest's Memory Concierge: persona + instructions, intent router, tool calls, short-term memory, Gradio UI.

**Sections:** 1 Setup · 2 Business Context · 3 🎭 Persona · 4 📏 Instructions · 5 System Prompt · 6 Tools + Schemas · 7 Router · 8 Agent Loop · 9 Orchestrator · 10 Reflection · 11 Tests · 12 Gradio

## 1. Setup

### 1.1 Imports and configuration
All paths and the model name live in one place so the same code works in this notebook, `app.py`, and Hugging Face Spaces.

In [1]:
import os
import sys
from pathlib import Path

print("Python :", sys.version.split()[0], "@", sys.executable)
try:
    import openai, gradio, pypdf, dotenv
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(f"{e}. Wrong kernel: select 'Python (business_bot)' / the .venv kernel, then restart.") from None

ME_DIR = Path("me")
LOGS_DIR = Path("logs")
LOGS_DIR.mkdir(exist_ok=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.5-flash-lite"   # confirm the exact current name in AI Studio

print("openai :", openai.__version__)
print("gradio :", gradio.__version__)
print("pypdf  :", pypdf.__version__)
print("cwd    :", os.getcwd())

Python : 3.12.6 @ C:\Graduate Studies\Agentic Systems\Assignment 3\business_bot\.venv\Scripts\python.exe


C:\Graduate Studies\Agentic Systems\Assignment 3\business_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


openai : 3.26.1
gradio : 6.29.1
pypdf  : 6.19.0
cwd    : C:\Graduate Studies\Agentic Systems\Assignment 3\business_bot


### 1.2 Load the API key
Works in both places: **Colab Secrets** when running in Colab, otherwise the local **`.env`** file (git-ignored).
The key is never printed in full.

In [2]:
try:
    from google.colab import userdata          # only exists in Colab
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import find_dotenv, load_dotenv   # local Jupyter / app.py
    load_dotenv(find_dotenv(usecwd=True))        # search from the working directory

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "API key not found — add GEMINI_API_KEY to .env (or Colab Secrets)"
print("✅ Key loaded:", GEMINI_API_KEY[:4] + "...")

✅ Key loaded: AQ.A...


### 1.3 LLM client and `call_llm()`
One client, created once, using the `openai` SDK pointed at Gemini's OpenAI-compatible endpoint (`GEMINI_BASE_URL`,
`MODEL` from 1.1). The SDK's own automatic retries are switched off (`max_retries=0`), so every request is visible and
counted. Each request uses part of the daily quota. `call_llm()` is the single entry point that the router, the agent
loop and reflection all build on. It returns the assistant **message** (`choices[0].message`), so later stages can read
both `.content` and `.tool_calls`. It retries once after an HTTP 429 (rate limit) or 5xx (e.g. 503 "model overloaded") and turns other API errors into readable messages.

In [3]:
import time
from openai import OpenAI

client = OpenAI(api_key=GEMINI_API_KEY, base_url=GEMINI_BASE_URL, max_retries=0)
LLM_CALLS = 0  # requests sent this session; each one counts against the daily quota


def call_llm(messages, tools=None, model=None, retry_wait=15, **kwargs):
    """Send one chat request and return the assistant message (resp.choices[0].message)."""
    global LLM_CALLS
    params = {"model": model or MODEL, "messages": messages, **kwargs}
    if tools:
        params["tools"] = tools

    for attempt in (1, 2):
        LLM_CALLS += 1
        try:
            resp = client.chat.completions.create(**params)
            return resp.choices[0].message
        except (openai.RateLimitError, openai.InternalServerError) as e:
            busy = isinstance(e, openai.InternalServerError)  # 5xx, e.g. 503 "model overloaded"
            if attempt == 2:
                if busy:
                    raise RuntimeError(f"Gemini is temporarily unavailable (HTTP {e.status_code}). "
                                       "Try again in a minute.") from None
                raise RuntimeError("Rate limit or daily quota reached (HTTP 429). Wait a minute and try again. "
                                   "Daily quotas reset at midnight Pacific time.") from None
            reason = "server busy" if busy else "rate limit"
            print(f"⏳ HTTP {e.status_code} ({reason}). Retrying once in {retry_wait}s...")
            time.sleep(retry_wait)
        except openai.NotFoundError:
            raise RuntimeError(f"Model {params['model']!r} not found (HTTP 404). "
                               "Check the exact model name in AI Studio.") from None
        except (openai.AuthenticationError, openai.PermissionDeniedError):
            raise RuntimeError("API key rejected (HTTP 401/403). Check GEMINI_API_KEY in .env.") from None
        except openai.BadRequestError as e:
            raise RuntimeError(f"Bad request (HTTP 400): {e.message}") from None
        except openai.APIConnectionError:
            raise RuntimeError("Could not reach the Gemini API. Check your internet connection.") from None

### 1.4 First LLM call: messages and roles
A chat request is just `model` + a list of `messages`. Each message has a **role** and some **content**:

| Role | Who writes it | Purpose |
|---|---|---|
| `system` | the developer (us) | Hidden instructions: persona, rules and business facts. The model reads it on every turn. |
| `user` | the customer | What they typed. |
| `assistant` | the model | Its earlier replies. If it contains `tool_calls`, the model is asking us to run a tool. |
| `tool` | our code | The result of a tool we ran, linked to the request by `tool_call_id`. Used from Stage 6. |

**Experiment 1: the raw response.** One call made directly with the client, so the whole response object is visible.
Look for `finish_reason` (why it stopped), `usage` (tokens billed), and `message.role`.

In [4]:
resp = client.chat.completions.create(model=MODEL, messages=[
    {"role": "system", "content": "You answer in one sentence."},
    {"role": "user", "content": "What is an API?"},
])
LLM_CALLS += 1
print(resp.model_dump_json(indent=2, exclude_none=True))

choice = resp.choices[0]
print("\nfinish_reason :", choice.finish_reason)
print("message.role  :", choice.message.role)
print("usage         :", resp.usage.prompt_tokens, "prompt +", resp.usage.completion_tokens, "completion tokens")
print("answer        :", choice.message.content)

{
  "id": "c-LHaovGFLHZ28oPvfnkmQ0",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and exchange data with each other.",
        "role": "assistant",
        "extra_content": {
          "google": {
            "thought_signature": "EmAKXgFpFH0TOYe+/uwtjierB7T8+7J9TtRAzRY3tUb81piXoywv4O8tvf2r6pXQhOgiqqeUdmYB8rD8f/pYqOjayDhxQC4KCfsKPdmsD57KhU6aXscUh4Ho/yAjlgF41nw="
          }
        }
      }
    }
  ],
  "created": 1791484541,
  "model": "gemini-3.5-flash-lite",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 29,
    "prompt_tokens": 13,
    "total_tokens": 42
  }
}

finish_reason : stop
message.role  : assistant
usage         : 13 prompt + 29 completion tokens
answer        : An API, or Application Programming Interface, is a set of rules and protocols that allows di

Note the `extra_content.google.thought_signature` field. Gemini attaches it to its replies, and during tool calling it
**must be sent back unchanged**. That's why the Stage 6 agent loop appends the assistant message exactly as returned
instead of rebuilding it by hand.

**Experiment 2: the system message controls behavior.** Same question, two different system messages.

In [5]:
for system in ["You answer in one sentence.", "Answer like a pirate, in one sentence."]:
    reply = call_llm([
        {"role": "system", "content": system},
        {"role": "user", "content": "What is an API?"},
    ]).content
    print(f"SYSTEM: {system}\n→ {reply}\n")

SYSTEM: You answer in one sentence.
→ An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and exchange data with each other.



SYSTEM: Answer like a pirate, in one sentence.
→ An API be like a magical message-in-a-bottle that lets two different ships talk and share treasure without boarding each other!



**Experiment 3: the model is stateless.** First, two *separate* calls: tell it a name, then ask for it. Then *one*
call that includes the earlier messages.

In [6]:
intro = [{"role": "user", "content": "My name is Sara."}]
question = {"role": "user", "content": "What is my name? Answer in one short sentence."}

first_reply = call_llm(intro).content
separate = call_llm([question]).content
together = call_llm(intro + [{"role": "assistant", "content": first_reply}, question]).content

print("Separate calls →", separate)
print("One call with history →", together)

assert "sara" not in separate.lower(), "unexpected: the model knew the name without history"
assert "sara" in together.lower(), "the model should recall the name when the history is sent"
print("\n✅ Statelessness confirmed")

Separate calls → I do not know your name because you haven't told me yet.
One call with history → Your name is Sara.

✅ Statelessness confirmed


**Why it "forgot":** the model keeps nothing between requests. Each call only sees the messages in that call's list.
The second call contained only "What is my name?", so the name simply wasn't there. When the earlier messages are
sent again, it "remembers". **That is all short-term memory is:** resending the chat history on every request.
Gradio will keep that history for us (Stage 8–9).

**Stage 3 checks:** `call_llm()` returns non-empty text, and a wrong model name gives a readable error instead of a
confusing traceback.

In [7]:
reply = call_llm([{"role": "user", "content": "Say hello in exactly three words."}]).content
assert isinstance(reply, str) and reply.strip(), "call_llm returned no text"
print("✅ call_llm returned:", reply)

try:
    call_llm([{"role": "user", "content": "hi"}], model="gemini-does-not-exist")
    raise AssertionError("expected a RuntimeError for a wrong model name")
except RuntimeError as e:
    print("✅ Readable error for a wrong model:", e)

print(f"\n📊 LLM requests sent this session: {LLM_CALLS}")

✅ call_llm returned: Hello to you.
✅ Readable error for a wrong model: Model 'gemini-does-not-exist' not found (HTTP 404). Check the exact model name in AI Studio.

📊 LLM requests sent this session: 8


## 2. Business Context

Milo may only answer from two files in `me/`: `business_summary.txt` and `about_business.pdf`.
The PDF is turned into a string with `pypdf` (`page.extract_text()` for each page), the pages are joined with blank
lines, and extra whitespace is cleaned up. Both files are small (a few thousand characters), so they go straight into
the system prompt in Section 5. A project this size doesn't need retrieval (RAG).

In [8]:
import re
from pypdf import PdfReader

BUSINESS_NAME = "EchoNest"


def clean_text(text):
    """Normalize line endings, collapse runs of spaces, trim each line, and allow at most one blank line."""
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_business_context(me_dir=ME_DIR):
    """Return (summary_text, pdf_text) loaded from me/business_summary.txt and me/about_business.pdf."""
    me_dir = Path(me_dir)
    summary_path = me_dir / "business_summary.txt"
    pdf_path = me_dir / "about_business.pdf"
    for path in (summary_path, pdf_path):
        if not path.exists():
            raise FileNotFoundError(f"{path} not found (cwd = {os.getcwd()}). Run from the repo root.")

    summary_text = clean_text(summary_path.read_text(encoding="utf-8"))
    reader = PdfReader(pdf_path)
    pdf_text = clean_text("\n\n".join((page.extract_text() or "") for page in reader.pages))
    return summary_text, pdf_text

In [9]:
summary_text, pdf_text = load_business_context()

for name, text in [("business_summary.txt", summary_text), ("about_business.pdf", pdf_text)]:
    print(f"📄 {name}: {len(text):,} chars (~{len(text) // 4:,} tokens)")
    print(text[:300], "...\n")

📄 business_summary.txt: 2,575 chars (~643 tokens)
EchoNest is a digital-memory studio. Our tagline is "Your memories, told your way." We turn a person's photos, voice recordings, written memories, important dates and captions into a personalized digital memory experience: a chronological story with captions, short narratives, key milestones, a visu ...

📄 about_business.pdf: 5,836 chars (~1,459 tokens)
EchoNest
Your memories, told your way.
1. Company Overview
EchoNest is a digital-memory studio. We transform a person's photos, voice recordings, written
memories, important dates and captions into a personalized digital memory experience. Instead
of leaving photos in a folder, we organize them into ...



### 2.1 Context checks (no API calls)
These use the Stage 1 test plan in `tests/knowledge_checks.json`. Whitespace is collapsed before matching, because
the PDF can break a phrase across two lines.

In [10]:
import json

checks = json.loads(Path("tests/knowledge_checks.json").read_text(encoding="utf-8"))


def flat(text):
    return re.sub(r"\s+", " ", text)


pdf_flat, summary_flat = flat(pdf_text), flat(summary_text)

assert len(pdf_text) > 500, "PDF text too short. Is the PDF a scanned image?"
assert len(summary_text) > 200, "Summary too short"
assert BUSINESS_NAME in pdf_text and BUSINESS_NAME in summary_text, "Business name missing"
for item in checks["pdf_only"]:
    assert item["phrase"] in pdf_flat, f"PDF-only fact missing: {item['phrase']!r}"
for item in checks["summary_only"]:
    assert item["phrase"] in summary_flat, f"Summary-only fact missing: {item['phrase']!r}"
for fact in checks["shared_facts"]:
    assert fact in pdf_flat and fact in summary_flat, f"Shared fact missing: {fact!r}"

total = len(pdf_text) + len(summary_text)
print(f"✅ Context OK: {len(checks['pdf_only'])} PDF-only, {len(checks['summary_only'])} summary-only and "
      f"{len(checks['shared_facts'])} shared facts found ({total:,} chars total, ~{total // 4:,} tokens)")

✅ Context OK: 6 PDF-only, 3 summary-only and 16 shared facts found (8,411 chars total, ~2,102 tokens)
